# Stack Overflow Scraper Tutorial

This notebook demonstrates how to build a comprehensive Stack Overflow scraper using the Stack Exchange API. We'll fetch questions and answers related to a specific tag (like "jax") and create structured Q&A pair compatible with the Gemini API.

## Overview

The scraper will:
1. Fetch questions from Stack Overflow with a specific tag
2. Get all answers for each question
3. Create structured Q&A records
4. Save the data as JSON for further processing

## Imports

In [3]:
import json
import os
import time
from typing import Any, Dict, List, Optional

import requests
from dotenv import load_dotenv
from tqdm import tqdm

## Config Setup

Let's set up our configuration constants. These control how our scraper behaves:

- **STACK_API_URL**: The base URL for the Stack Exchange API
- **TAG**: The tag we want to filter questions by (e.g., "jax")
- **MIN_VOTES**: Minimum votes required for questions/answers to be included
- **MAX_QUESTIONS_TO_FETCH**: How many questions to fetch
- **QUESTIONS_PER_PAGE**: Number of questions per API call
- **REQUEST_DELAY_SECONDS**: Delay between API calls to be respectful

In [11]:
assert load_dotenv(), "Couldn't load envvars"

STACK_API_URL: str = "https://api.stackexchange.com/2.3"
TAG: str = "python"
MIN_VOTES: int = 0
MAX_QUESTIONS_TO_FETCH: int = 50  # Approx number of questions to fetch
QUESTIONS_PER_PAGE: int = 25
REQUEST_DELAY_SECONDS: float = 0.5  # Delay between API calls to be polite

OUTPUT_DIR: str = "data"
OUTPUT_FILENAME: str = os.path.join(OUTPUT_DIR, f"so_{TAG.lower()}_qa_pairs.json")

print(f"Configuration loaded:")
print(f"- Tag: {TAG}")
print(f"- Max questions: {MAX_QUESTIONS_TO_FETCH}")
print(f"- Output file: {OUTPUT_FILENAME}")

Configuration loaded:
- Tag: python
- Max questions: 50
- Output file: data/so_python_qa_pairs.json


## Making requests

This is the foundation of our scraper - a robust function to make requests to the Stack Exchange API. It handles:

- Error handling for network issues
- API error responses
- Rate limiting (quota and backoff)
- Timeouts

In [12]:
def make_stack_api_request(endpoint: str, params: Dict[str, Any]) -> Dict[str, Any]:
    """
    Makes a request to the Stack Exchange API.

    Args:
        endpoint: The API endpoint (e.g., "/questions").
        params: A dictionary of parameters for the API call.

    Returns:
        The JSON response from the API.

    Raises:
        requests.exceptions.RequestException: If the HTTP request fails.
        ValueError: If the API returns an error.
    """
    url: str = f"{STACK_API_URL}{endpoint}"

    # Always specify that we're querying Stack Overflow specifically
    params["site"] = "stackoverflow"

    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()  # Raises an HTTPError for bad responses (4XX or 5XX)
    except requests.exceptions.Timeout:
        print(f"Timeout occurred while requesting {url} with params {params}")
        raise
    except requests.exceptions.RequestException as e:
        print(f"Request to {url} failed: {e}")
        print(
            f"Response content: {response.content if 'response' in locals() else 'No response object'}"
        )
        raise

    data: Dict[str, Any] = response.json()

    # Check for API-level errors
    if data.get("error_id"):
        raise ValueError(
            f"Stack API error {data.get('error_id')}: {data.get('error_name')} - {data.get('error_message')}"
        )

    # Handle rate limiting
    if "quota_remaining" in data and data["quota_remaining"] < 10:  # Be conservative
        print(f"Warning: Low API quota remaining: {data['quota_remaining']}")
    if data.get("backoff"):
        backoff_seconds = data["backoff"]
        print(f"API requested backoff for {backoff_seconds} seconds. Waiting...")
        time.sleep(backoff_seconds + 1)  # Add a small buffer

    return data

## Testing the API

Let's test our API function with a simple request to make sure everything is working:

In [13]:
try:
    test_params = {
        "page": 1,
        "pagesize": 1,
        "order": "desc",
        "sort": "votes",
        "tagged": TAG,
        "filter": "withbody"
    }
    test_response = make_stack_api_request("/questions", test_params)
    print(f"✅ API test successful!")
    print(f"Quota remaining: {test_response.get('quota_remaining', 'Unknown')}")
    print(f"Found {len(test_response.get('items', []))} questions in test")
except Exception as e:
    print(f"❌ API test failed: {e}")

✅ API test successful!
Quota remaining: 298
Found 1 questions in test


## Fetching Questions

Now let's create a function to fetch multiple questions with pagination. This function:

- Handles pagination to get more than one page of results
- Uses a progress bar to show download progress
- Sorts questions by votes to get higher quality content first
- Includes the question body text with the `withbody` filter

In [14]:
def get_questions(tag: str, num_questions: int, page_size: int) -> List[Dict[str, Any]]:
    """
    Fetches questions from Stack Overflow with a specific tag.

    Args:
        tag: The tag to filter questions by (e.g., "jax").
        num_questions: The approximate total number of questions to fetch.
        page_size: Number of questions to fetch per API call.

    Returns:
        A list of question objects from the API.
    """
    all_questions: List[Dict[str, Any]] = []
    page: int = 1
    fetched_count: int = 0

    print(f"Fetching questions tagged '{tag}'...")
    with tqdm(total=num_questions, unit="question") as pbar:
        while fetched_count < num_questions:
            params: Dict[str, Any] = {
                "page": page,
                "pagesize": min(
                    page_size, num_questions - fetched_count
                ),  # Adjust last page size
                "order": "desc",
                "sort": "votes",  # Sort by votes to get potentially higher quality questions first
                "tagged": tag,
                "filter": "withbody",  # Includes question body
            }
            try:
                data: Dict[str, Any] = make_stack_api_request("/questions", params)
                questions_on_page: List[Dict[str, Any]] = data.get("items", [])
                all_questions.extend(questions_on_page)

                newly_fetched = len(questions_on_page)
                fetched_count += newly_fetched
                pbar.update(newly_fetched)

                if not data.get("has_more") or not questions_on_page:
                    print("No more questions found or API limit reached.")
                    break
                page += 1
                time.sleep(REQUEST_DELAY_SECONDS)  # Polite delay
            except (requests.exceptions.RequestException, ValueError) as e:
                print(f"Error fetching questions on page {page}: {e}")
                # Decide if you want to retry or break. For simplicity, we break here.
                break
            if fetched_count >= num_questions:
                break

    # Ensure we don't exceed the requested number significantly due to page sizes
    return all_questions[:num_questions]

## Fetching Answers for Questions

For each question, we need to fetch all its answers. This function:

- Takes a question ID and fetches all associated answers
- Handles pagination for questions with many answers
- Sorts answers by votes to prioritize better answers
- Includes answer body text

In [15]:
def get_answers_for_question(question_id: int) -> List[Dict[str, Any]]:
    """
    Fetches all answers for a given Stack Overflow question ID.

    Args:
        question_id: The ID of the question.

    Returns:
        A list of answer objects from the API.
    """
    all_answers: List[Dict[str, Any]] = []
    page: int = 1

    print(f"Fetching answers for question ID: {question_id}...")
    while True:
        params: Dict[str, Any] = {
            "page": page,
            "pagesize": 100,  # Max page size for answers
            "order": "desc",
            "sort": "votes",
            "filter": "withbody",  # Includes answer body
        }
        try:
            data: Dict[str, Any] = make_stack_api_request(
                f"/questions/{question_id}/answers", params
            )
            answers_on_page: List[Dict[str, Any]] = data.get("items", [])
            all_answers.extend(answers_on_page)

            if not data.get("has_more") or not answers_on_page:
                break
            page += 1
            time.sleep(REQUEST_DELAY_SECONDS)  # Polite delay
        except (requests.exceptions.RequestException, ValueError) as e:
            print(
                f"Error fetching answers for question {question_id} on page {page}: {e}"
            )
            break  # Stop trying for this question if an error occurs

    return all_answers

## Creating Structured Q&A Records

This function takes raw question and answer data from the API and creates structured records suitable for machine learning training. Each record contains:

- Question text as input
- Answer text as output
- Metadata like votes, IDs, and links
- User information

We also filter out questions and answers that don't meet our minimum vote threshold.

In [16]:
def create_qa_records(
    question: Dict[str, Any], answers: List[Dict[str, Any]]
) -> List[Dict[str, Any]]:
    """
    Creates serialized Q&A records from a question and its answers.

    Args:
        question: The question object from the API.
        answers: A list of answer objects for the question.

    Returns:
        A list of Q&A records in the desired format.
    """
    qa_records: List[Dict[str, Any]] = []
    question_votes: int = question.get("score", 0)
    question_owner_id: Optional[str] = str(
        question.get("owner", {}).get("user_id", "N/A")
    )
    question_body_html: str = question.get("body", "")
    question_id: int = question.get("question_id", -1)
    question_title: str = question.get("title", "N/A")
    question_link: str = question.get("link", "")

    if question_votes < MIN_VOTES:
        return []  # Skip if question itself doesn't meet vote criteria

    for answer in answers:
        answer_votes: int = answer.get("score", 0)
        if answer_votes < MIN_VOTES:
            continue  # Skip if answer doesn't meet vote criteria

        answer_owner_id: Optional[str] = str(
            answer.get("owner", {}).get("user_id", "N/A")
        )
        answer_body_html: str = answer.get("body", "")
        answer_id: int = answer.get("answer_id", -1)
        answer_link: str = (
            f"{question_link}/{answer_id}#{answer_id}"  # Construct answer link
        )

        record: Dict[str, Any] = {
            "text_input": question_body_html,
            "output": answer_body_html,
            "from_id": question_owner_id,
            "to_id": answer_owner_id,
            "answer_votes": answer_votes,
            "question_votes": question_votes,
            "question_id": question_id,
            "answer_id": answer_id,
            "question_title": question_title,
            "question_link": question_link,
            "answer_link": answer_link,
        }
        qa_records.append(record)
    return qa_records

## Main Orchestration Function

This is the main function that ties everything together. It:

1. Creates the output directory if it doesn't exist
2. Fetches questions using our `get_questions` function
3. For each question, fetches its answers
4. Creates structured Q&A records
5. Saves all data to a JSON file

Let's break this into smaller parts for better understanding:

In [17]:
def main() -> None:
    """
    Main function to orchestrate fetching Q&A pairs and saving them.
    """
    # Step 1: Create output directory
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
        print(f"Created output directory: {OUTPUT_DIR}")

    # Step 2: Fetch questions
    questions: List[Dict[str, Any]] = get_questions(
        TAG, MAX_QUESTIONS_TO_FETCH, QUESTIONS_PER_PAGE
    )

    all_qa_data: List[Dict[str, Any]] = []

    if not questions:
        print(f"No questions found for tag '{TAG}'. Exiting.")
        return

    # Step 3: Process each question to get answers and create Q&A pairs
    print(f"\nProcessing {len(questions)} questions to find Q&A pairs...")
    for question in tqdm(questions, unit="question processed"):
        question_id: Optional[int] = question.get("question_id")
        if question_id is None:
            print(f"Skipping question with missing ID: {question.get('title')}")
            continue

        answers: List[Dict[str, Any]] = get_answers_for_question(question_id)
        if answers:
            qa_records: List[Dict[str, Any]] = create_qa_records(question, answers)
            all_qa_data.extend(qa_records)
        time.sleep(
            REQUEST_DELAY_SECONDS
        )  # Delay even if no answers, for the question fetch itself

    print(f"\nCollected {len(all_qa_data)} Q&A pairs.")

    # Step 4: Save the data
    if all_qa_data:
        try:
            with open(OUTPUT_FILENAME, "w", encoding="utf-8") as f:
                json.dump(all_qa_data, f, indent=4, ensure_ascii=False)
            print(f"Successfully saved Q&A pairs to: {OUTPUT_FILENAME}")
        except IOError as e:
            print(f"Error saving data to {OUTPUT_FILENAME}: {e}")
    else:
        print("No Q&A pairs collected that meet the criteria.")

## Running the Scraper

Now let's run our complete scraper! This will:

1. Fetch questions tagged with our specified tag
2. Get answers for each question
3. Create structured Q&A pairs
4. Save everything to a JSON file

**Note**: This may take a while depending on how many questions you're fetching and the API response times.

In [18]:
main()

Created output directory: data
Fetching questions tagged 'python'...


100%|██████████| 50/50 [00:01<00:00, 31.98question/s]



Processing 50 questions to find Q&A pairs...


  0%|          | 0/50 [00:00<?, ?question processed/s]

Fetching answers for question ID: 231767...


...

Fetching answers for question ID: 2720014...


 98%|█████████▊| 49/50 [00:39<00:00,  1.27question processed/s]

Fetching answers for question ID: 72899...


100%|██████████| 50/50 [00:40<00:00,  1.23question processed/s]


Collected 1426 Q&A pairs.
Successfully saved Q&A pairs to: data/so_python_qa_pairs.json


## Analyzing the Results

Let's examine what we've collected by loading and analyzing our output file:

In [19]:
# Load and analyze the results
if os.path.exists(OUTPUT_FILENAME):
    with open(OUTPUT_FILENAME, "r", encoding="utf-8") as f:
        qa_data = json.load(f)
    
    print(f"📊 Analysis of collected data:")
    print(f"Total Q&A pairs: {len(qa_data)}")
    
    if qa_data:
        # Analyze vote distributions
        question_votes = [item['question_votes'] for item in qa_data]
        answer_votes = [item['answer_votes'] for item in qa_data]
        
        print(f"\n📈 Vote statistics:")
        print(f"Question votes - Min: {min(question_votes)}, Max: {max(question_votes)}, Avg: {sum(question_votes)/len(question_votes):.1f}")
        print(f"Answer votes - Min: {min(answer_votes)}, Max: {max(answer_votes)}, Avg: {sum(answer_votes)/len(answer_votes):.1f}")
        
        # Show a sample record
        print(f"\n📝 Sample Q&A pair:")
        sample = qa_data[0]
        print(f"Question title: {sample['question_title']}")
        print(f"Question votes: {sample['question_votes']}")
        print(f"Answer votes: {sample['answer_votes']}")
        print(f"Question preview: {sample['text_input'][:200]}...")
        print(f"Answer preview: {sample['output'][:200]}...")
else:
    print(f"❌ Output file {OUTPUT_FILENAME} not found. Run the scraper first.")

📊 Analysis of collected data:
Total Q&A pairs: 1426

📈 Vote statistics:
Question votes - Min: 2850, Max: 13087, Avg: 4663.5
Answer votes - Min: 0, Max: 18231, Avg: 303.8

📝 Sample Q&A pair:
Question title: What does the &quot;yield&quot; keyword do in Python?
Question votes: 13087
Answer votes: 18231
Question preview: <p>What functionality does the <a href="https://docs.python.org/3/reference/simple_stmts.html#yield" rel="noreferrer"><code>yield</code></a> keyword in Python provide?</p>
<p>For example, I'm trying t...
Answer preview: <p>To understand what <a href="https://docs.python.org/3/reference/simple_stmts.html#yield" rel="noreferrer"><code>yield</code></a> does, you must understand what <em><a href="https://docs.python.org/...


## Conclusion

Congratulations! You've successfully built a decent Stack Overflow scraper

### Potential Improvements:
- Add retry logic for failed requests
- Implement data deduplication
- Add support for multiple tags
- Include comment data
- Add data validation and cleaning

### Usage Tips:
- Adjust `MAX_QUESTIONS_TO_FETCH` based on your needs
- Increase `MIN_VOTES` to get higher quality content
- Monitor your API quota to avoid hitting limits
- Consider running during off-peak hours for better performance